# Milestone 2 — Transformers

**Objective:** Use pretrained transformer models to generate context-aware embeddings and improve MCQ answer ranking over the TF-IDF baseline.

**Key Concepts Covered:**
- HuggingFace `transformers` and `sentence-transformers` libraries
- BERT/RoBERTa architecture and attention mechanisms
- Pretrained embedding models for semantic similarity
- Zero-shot classification with transformer models

**Models I tried:**
1. Sentence-Transformers (all-MiniLM-L6-v2) + cosine similarity
2. A larger sentence-transformer (all-mpnet-base-v2) + cosine similarity
3. Zero-shot classification (BART-large-MNLI / DeBERTa-v3-base-mnli-fever-anli)
4. BERT [CLS] token approach for prompt-option pair scoring

**Expected improvement:** MAP@3 from ~0.30 (TF-IDF baseline) to ~0.50–0.65

**Note:** This notebook requires GPU. On Kaggle: Settings → Accelerator → GPU T4 x2

In [1]:
!pip install sentence-transformers wandb -q

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import wandb
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 50.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

## 1. Load the Dataset

The competition dataset contains MCQ-style questions. Each row has:
- `id` — unique question identifier
- `prompt` — the question text
- `A`, `B`, `C`, `D`, `E` — five answer option texts
- `answer` — the correct answer label (only in train set)

We load both `train.csv` (for building and evaluating models) and `test.csv` (for Kaggle submission).

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

option_cols = ['A', 'B', 'C', 'D', 'E']

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

def ap_at_3(true_label, predicted_labels):
    for i, pred in enumerate(predicted_labels[:3]):
        if pred.strip().upper() == true_label.strip().upper():
            return 1.0 / (i + 1)
    return 0.0

def map_at_3(true_labels, predicted_labels):
    scores = [ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)]
    return np.mean(scores)

def map_at_3_detailed(true_labels, predicted_labels):
    scores = [ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)]
    n = len(scores)
    return {
        'map3': np.mean(scores),
        'correct_at_1': sum(1 for s in scores if s == 1.0),
        'correct_at_2': sum(1 for s in scores if s == 0.5),
        'correct_at_3': sum(1 for s in scores if abs(s - 1/3) < 0.01),
        'missed': sum(1 for s in scores if s == 0.0),
        'total': n,
        'top1_acc': sum(1 for s in scores if s == 1.0) / n,
        'top3_acc': sum(1 for s in scores if s > 0) / n,
    }

def print_results(name, results):
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  MAP@3:          {results['map3']:.4f}")
    print(f"  Top-1 Accuracy: {results['top1_acc']:.2%}")
    print(f"  Top-3 Accuracy: {results['top3_acc']:.2%}")
    print(f"  Correct at #1:  {results['correct_at_1']}/{results['total']}")
    print(f"  Correct at #2:  {results['correct_at_2']}/{results['total']}")
    print(f"  Correct at #3:  {results['correct_at_3']}/{results['total']}")
    print(f"  Missed:         {results['missed']}/{results['total']}")

print("Setup complete.")

Train: (2000, 8), Test: (500, 7)
Setup complete.


## 2. W&B Login

Login to Weights & Biases for experiment tracking. API key is stored as a Kaggle Secret named `WANDB_API_KEY`.

In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))
print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B login successful!


## 3. Model 1 — Sentence-Transformer (all-MiniLM-L6-v2)

**Sentence-Transformers** are pretrained models designed to produce meaningful sentence-level embeddings. Unlike raw BERT, they're fine-tuned so that semantically similar sentences produce similar embeddings.

**`all-MiniLM-L6-v2`** — lightweight (80MB), produces 384-dimensional embeddings. Fast and effective.

**How it differs from TF-IDF:**
- TF-IDF matches exact words — "automobile" and "car" get zero similarity
- Sentence-transformers understand meaning — "automobile" and "car" get high similarity
- Sentence-transformers capture word order — "dog bites man" ≠ "man bites dog"

In [4]:
model_mini = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model_mini.get_sentence_embedding_dimension()}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


### 3.1 Encode All Texts & Predict

We encode every prompt and every option into dense vectors in batches, then rank options by cosine similarity to the prompt.

In [5]:
def predict_sbert(df, model, batch_size=64):
    """
    Predict top-3 answers using sentence-transformer embeddings.
    Encodes all prompts and options in batches for efficiency.
    """
    prompt_embeddings = model.encode(
        df['prompt'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    option_embeddings = {}
    for col in option_cols:
        option_embeddings[col] = model.encode(
            df[col].astype(str).tolist(),
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True
        )

    predictions = []
    for idx in range(len(df)):
        prompt_vec = prompt_embeddings[idx].reshape(1, -1)

        similarities = {}
        for col in option_cols:
            opt_vec = option_embeddings[col][idx].reshape(1, -1)
            similarities[col] = cosine_similarity(prompt_vec, opt_vec)[0][0]

        ranked = sorted(similarities, key=similarities.get, reverse=True)
        predictions.append(ranked[:3])

    return predictions

print("Encoding and predicting with all-MiniLM-L6-v2...")
train_preds_mini = predict_sbert(train_df, model_mini)

results_mini = map_at_3_detailed(train_df['answer'].tolist(), train_preds_mini)
print_results("Sentence-Transformer (all-MiniLM-L6-v2)", results_mini)

Encoding and predicting with all-MiniLM-L6-v2...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Sentence-Transformer (all-MiniLM-L6-v2)
  MAP@3:          0.4231
  Top-1 Accuracy: 26.10%
  Top-3 Accuracy: 64.70%
  Correct at #1:  522/2000
  Correct at #2:  401/2000
  Correct at #3:  371/2000
  Missed:         706/2000


## 4. Model 2 — Sentence-Transformer (all-mpnet-base-v2)

**`all-mpnet-base-v2`** — larger model (420MB), produces 768-dimensional embeddings. Generally outperforms MiniLM on most benchmarks at the cost of being slower.

Comparing two sentence-transformer sizes helps us understand the accuracy vs compute trade-off.

In [6]:
model_mpnet = SentenceTransformer('all-mpnet-base-v2', device=device)
print(f"Model loaded: all-mpnet-base-v2")
print(f"Embedding dimension: {model_mpnet.get_sentence_embedding_dimension()}")

print("\nEncoding and predicting with all-mpnet-base-v2...")
train_preds_mpnet = predict_sbert(train_df, model_mpnet)

results_mpnet = map_at_3_detailed(train_df['answer'].tolist(), train_preds_mpnet)
print_results("Sentence-Transformer (all-mpnet-base-v2)", results_mpnet)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: all-mpnet-base-v2
Embedding dimension: 768

Encoding and predicting with all-mpnet-base-v2...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]


Sentence-Transformer (all-mpnet-base-v2)
  MAP@3:          0.4244
  Top-1 Accuracy: 25.90%
  Top-3 Accuracy: 65.75%
  Correct at #1:  518/2000
  Correct at #2:  391/2000
  Correct at #3:  406/2000
  Missed:         685/2000


## 5. Model 3 — Zero-Shot Classification

**Zero-shot classification** frames the task as Natural Language Inference (NLI):
- **Premise:** the question prompt
- **Hypothesis:** each answer option
- The model predicts whether each option **entails** (supports), **contradicts**, or is **neutral** to the prompt

**Why this is different from similarity:**
- Sentence-transformers measure how **alike** the prompt and option are
- Zero-shot measures whether the option logically **follows from** the prompt
- For MCQs, entailment is often more meaningful since the correct answer "answers" the question

We use `cross-encoder/nli-deberta-v3-base` which is trained on multiple NLI datasets.

In [7]:
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-base",
    device=0 if device == 'cuda' else -1
)

print("Zero-shot classifier loaded: cross-encoder/nli-deberta-v3-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Zero-shot classifier loaded: cross-encoder/nli-deberta-v3-base


### 5.1 Zero-Shot Prediction

For each question, we pass the prompt as the sequence and all 5 option texts as candidate labels. The model scores each option based on entailment probability and we take the top 3.

**Note:** This is slower than embedding approaches since the model processes each prompt-option pair individually. Expect ~15–30 minutes on T4.

In [8]:
def predict_zero_shot(df, clf, max_rows=None):
    """Predict using zero-shot classification."""
    predictions = []
    n = len(df) if max_rows is None else min(max_rows, len(df))

    for idx in range(n):
        if idx % 100 == 0:
            print(f"  Processing {idx}/{n}...")

        row = df.iloc[idx]
        prompt = row['prompt']

        candidate_labels = [str(row[col]) for col in option_cols]

        result = clf(prompt, candidate_labels, multi_label=False)

        label_scores = {}
        for label, score in zip(result['labels'], result['scores']):
            for col in option_cols:
                if str(row[col]) == label:
                    label_scores[col] = score
                    break

        ranked = sorted(label_scores, key=label_scores.get, reverse=True)
        predictions.append(ranked[:3])

    return predictions

print("Running zero-shot classification on train set...")
train_preds_zs = predict_zero_shot(train_df, classifier)

results_zs = map_at_3_detailed(train_df['answer'].tolist(), train_preds_zs)
print_results("Zero-Shot Classification (DeBERTa-v3-base NLI)", results_zs)

Running zero-shot classification on train set...
  Processing 0/2000...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Processing 100/2000...
  Processing 200/2000...
  Processing 300/2000...
  Processing 400/2000...
  Processing 500/2000...
  Processing 600/2000...
  Processing 700/2000...
  Processing 800/2000...
  Processing 900/2000...
  Processing 1000/2000...
  Processing 1100/2000...
  Processing 1200/2000...
  Processing 1300/2000...
  Processing 1400/2000...
  Processing 1500/2000...
  Processing 1600/2000...
  Processing 1700/2000...
  Processing 1800/2000...
  Processing 1900/2000...

Zero-Shot Classification (DeBERTa-v3-base NLI)
  MAP@3:          0.5624
  Top-1 Accuracy: 39.20%
  Top-3 Accuracy: 78.95%
  Correct at #1:  784/2000
  Correct at #2:  455/2000
  Correct at #3:  340/2000
  Missed:         421/2000


## 6. Model 4 — BERT [CLS] Token Approach

Here we use **BERT directly** (not sentence-transformers, not zero-shot) to score prompt-option pairs.

**How it works:**
1. For each option, concatenate: `[CLS] prompt [SEP] option [SEP]`
2. Pass through BERT to get the `[CLS]` hidden state
3. Compute the L2 norm of each `[CLS]` vector as a proxy score
4. Rank options by score

This captures **cross-attention** between prompt and option — unlike sentence-transformers which encode them independently.

**Limitation:** Using the raw [CLS] norm as a ranking signal is a rough heuristic. Fine-tuning (Milestone 4) adds a proper classification head to learn correct scoring.

In [9]:
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()

print("BERT-base-uncased loaded.")

def predict_bert_cls(df, tokenizer, model, max_rows=None):
    """Score each prompt-option pair using BERT's [CLS] token."""
    predictions = []
    n = len(df) if max_rows is None else min(max_rows, len(df))

    for idx in range(n):
        if idx % 200 == 0:
            print(f"  Processing {idx}/{n}...")

        row = df.iloc[idx]
        prompt = str(row['prompt'])

        scores = {}
        for col in option_cols:
            option_text = str(row[col])

            inputs = tokenizer(
                prompt,
                option_text,
                return_tensors='pt',
                truncation=True,
                max_length=512,
                padding=True
            ).to(device)

            with torch.no_grad():
                outputs = model(**inputs)

            cls_embedding = outputs.last_hidden_state[:, 0, :]
            scores[col] = torch.norm(cls_embedding).item()

        ranked = sorted(scores, key=scores.get, reverse=True)
        predictions.append(ranked[:3])

    return predictions

print("Running BERT [CLS] predictions on train set...")
train_preds_bert = predict_bert_cls(train_df, bert_tokenizer, bert_model)

results_bert = map_at_3_detailed(train_df['answer'].tolist(), train_preds_bert)
print_results("BERT [CLS] Token Approach", results_bert)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT-base-uncased loaded.
Running BERT [CLS] predictions on train set...
  Processing 0/2000...
  Processing 200/2000...
  Processing 400/2000...
  Processing 600/2000...
  Processing 800/2000...
  Processing 1000/2000...
  Processing 1200/2000...
  Processing 1400/2000...
  Processing 1600/2000...
  Processing 1800/2000...

BERT [CLS] Token Approach
  MAP@3:          0.4019
  Top-1 Accuracy: 26.05%
  Top-3 Accuracy: 59.40%
  Correct at #1:  521/2000
  Correct at #2:  363/2000
  Correct at #3:  304/2000
  Missed:         812/2000


## 7. Model Comparison

Side-by-side comparison of all 4 transformer approaches plus the TF-IDF baseline from Milestone 1.

In [10]:
comparison = pd.DataFrame({
    'Model': [
        'TF-IDF (M1 baseline)',
        'MiniLM-L6-v2',
        'mpnet-base-v2',
        'Zero-Shot (DeBERTa NLI)',
        'BERT [CLS]'
    ],
    'MAP@3': [
        0.2962,
        results_mini['map3'],
        results_mpnet['map3'],
        results_zs['map3'],
        results_bert['map3']
    ],
    'Top-1 Acc': [
        0.1355,
        results_mini['top1_acc'],
        results_mpnet['top1_acc'],
        results_zs['top1_acc'],
        results_bert['top1_acc']
    ],
    'Top-3 Acc': [
        0.4965,
        results_mini['top3_acc'],
        results_mpnet['top3_acc'],
        results_zs['top3_acc'],
        results_bert['top3_acc']
    ]
})

comparison = comparison.sort_values('MAP@3', ascending=False).reset_index(drop=True)

display_df = comparison.copy()
display_df['MAP@3'] = display_df['MAP@3'].apply(lambda x: f"{x:.4f}")
display_df['Top-1 Acc'] = display_df['Top-1 Acc'].apply(lambda x: f"{x:.2%}")
display_df['Top-3 Acc'] = display_df['Top-3 Acc'].apply(lambda x: f"{x:.2%}")

print(display_df.to_string(index=False))

                  Model  MAP@3 Top-1 Acc Top-3 Acc
Zero-Shot (DeBERTa NLI) 0.5624    39.20%    78.95%
          mpnet-base-v2 0.4244    25.90%    65.75%
           MiniLM-L6-v2 0.4231    26.10%    64.70%
             BERT [CLS] 0.4019    26.05%    59.40%
   TF-IDF (M1 baseline) 0.2962    13.55%    49.65%


## 8. Log All Runs to W&B

Each model is logged as a separate W&B run for comparison alongside Milestone 1 baselines.

In [11]:
PROJECT_NAME = "22f3002548-t22026"

# --- Run 1: MiniLM ---
wandb.init(project=PROJECT_NAME, name="m2-minilm-l6-v2", tags=["milestone2", "sentence-transformer"])
wandb.log({
    "model": "all-MiniLM-L6-v2",
    "map3": results_mini['map3'],
    "top1_accuracy": results_mini['top1_acc'],
    "top3_accuracy": results_mini['top3_acc'],
    "missed": results_mini['missed'],
    "embedding_dim": 384,
})
wandb.finish()

# --- Run 2: mpnet ---
wandb.init(project=PROJECT_NAME, name="m2-mpnet-base-v2", tags=["milestone2", "sentence-transformer"])
wandb.log({
    "model": "all-mpnet-base-v2",
    "map3": results_mpnet['map3'],
    "top1_accuracy": results_mpnet['top1_acc'],
    "top3_accuracy": results_mpnet['top3_acc'],
    "missed": results_mpnet['missed'],
    "embedding_dim": 768,
})
wandb.finish()

# --- Run 3: Zero-shot ---
wandb.init(project=PROJECT_NAME, name="m2-zero-shot-deberta", tags=["milestone2", "zero-shot"])
wandb.log({
    "model": "nli-deberta-v3-base",
    "map3": results_zs['map3'],
    "top1_accuracy": results_zs['top1_acc'],
    "top3_accuracy": results_zs['top3_acc'],
    "missed": results_zs['missed'],
})
wandb.finish()

# --- Run 4: BERT CLS ---
wandb.init(project=PROJECT_NAME, name="m2-bert-cls", tags=["milestone2", "bert"])
wandb.log({
    "model": "bert-base-uncased-cls",
    "map3": results_bert['map3'],
    "top1_accuracy": results_bert['top1_acc'],
    "top3_accuracy": results_bert['top3_acc'],
    "missed": results_bert['missed'],
    "embedding_dim": 768,
})
wandb.finish()

print("All 4 runs logged to W&B!")

wandb: setting up run cvgrinb9
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260720_180330-cvgrinb9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run m2-minilm-l6-v2
wandb: ⭐️ View project at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026
wandb: 🚀 View run at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026/runs/cvgrinb9
wandb: updating run metadata; uploading summary
wandb: uploading summary; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: uploading summary; uploading wandb-metadata.json; uploading requirements.txt; uploading config.yaml
wandb: uploading summary
wandb: uploading data
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: embedding_dim ▁
wandb:          map3 ▁
wandb:        missed ▁
wandb: top1_accuracy ▁
wandb: top3_accuracy ▁
wandb: 
wandb: Run summary:
wandb: embed

All 4 runs logged to W&B!


## 9. Generate Kaggle Submission

We use the best performing sentence-transformer model to generate test predictions and create the submission file.

In [12]:
models = {
    'minilm': (results_mini['map3'], model_mini),
    'mpnet': (results_mpnet['map3'], model_mpnet),
}

best_name = max(models, key=lambda k: models[k][0])
best_score, best_model = models[best_name]
print(f"Best model: {best_name} with MAP@3 = {best_score:.4f}")

print(f"\nGenerating test predictions with {best_name}...")
test_preds = predict_sbert(test_df, best_model)

submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': [' '.join(pred) for pred in test_preds]
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head())

submission.to_csv('submission.csv', index=False)
print("\nSaved to submission.csv")

Best model: mpnet with MAP@3 = 0.4244

Generating test predictions with mpnet...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Submission shape: (500, 2)
   id prediction
0   1      B A D
1   2      E B A
2   3      C A D
3   4      A C E
4   5      B A C

Saved to submission.csv


## 10. Error Analysis

Let's examine where the best model fails to understand its limitations and inform what we need to address in later milestones.

In [13]:
best_preds = train_preds_mpnet if best_name == 'mpnet' else train_preds_mini
train_scores = [ap_at_3(t, p) for t, p in zip(train_df['answer'].tolist(), best_preds)]
train_df['ap3_score'] = train_scores

missed_df = train_df[train_df['ap3_score'] == 0.0]
correct_df = train_df[train_df['ap3_score'] == 1.0]

print(f"Perfectly correct (score=1.0): {len(correct_df)}/{len(train_df)}")
print(f"Missed entirely (score=0.0):   {len(missed_df)}/{len(train_df)}")

print(f"\n{'='*60}")
print("SAMPLE FAILURES:")
print(f"{'='*60}")

for i, (_, row) in enumerate(missed_df.head(3).iterrows()):
    print(f"\nQ{i+1}: {str(row['prompt'])[:150]}...")
    print(f"  Correct: {row['answer']} = {str(row[row['answer']])[:80]}...")
    idx = row.name
    pred = best_preds[idx]
    print(f"  Predicted: {' '.join(pred)}")
    print(f"  Top pick: {pred[0]} = {str(row[pred[0]])[:80]}...")

train_df['prompt_len'] = train_df['prompt'].str.len()
print(f"\n{'='*60}")
print("Prompt length vs accuracy:")
print(f"{'='*60}")
print(f"  Correct answers — avg prompt length: {correct_df['prompt'].str.len().mean():.0f} chars")
print(f"  Missed answers  — avg prompt length: {missed_df['prompt'].str.len().mean():.0f} chars")

Perfectly correct (score=1.0): 518/2000
Missed entirely (score=0.0):   685/2000

SAMPLE FAILURES:

Q1: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options....
  Correct: B = Martin Heidegger believes that humans do not exist inside time, but that they ar...
  Predicted: D C E
  Top pick: D = Martin Heidegger believes that the relationship between time and human existence...

Q2: Select the most accurate option: What is the Peierls bracket in canonical quantization? based on the given context....
  Correct: C = The Peierls bracket is a Poisson bracket derived from the action in the canonica...
  Predicted: B D A
  Top pick: B = The Peierls bracket is a mathematical tool used to generate the Hamiltonian in t...

Q3: Choose the correct answer: What is the throttling process, and why is it essential? from the following choices....
  Correct: B = The throttling process is a steady adiabatic flow of a f

In [14]:
# Verify submission file
submission = pd.read_csv('submission.csv')
print(f"Submission shape: {submission.shape}")
print(f"Columns: {list(submission.columns)}")
print(f"Sample predictions:")
print(submission.head(10))

# Sanity checks
assert submission.shape[0] == len(test_df), "Row count mismatch!"
assert all(len(p.split()) == 3 for p in submission['prediction']), "All predictions must have exactly 3 labels!"
print(f"\n✓ All {len(submission)} rows have exactly 3 predictions. Ready to submit!")

Submission shape: (500, 2)
Columns: ['id', 'prediction']
Sample predictions:
   id prediction
0   1      B A D
1   2      E B A
2   3      C A D
3   4      A C E
4   5      B A C
5   6      A D C
6   7      E D A
7   8      C A D
8   9      A C D
9  10      B E C

✓ All 500 rows have exactly 3 predictions. Ready to submit!
